# Parihaka facies: U-Net training on Colab
Run the cells top to bottom. First set **Runtime > Change runtime type > T4 GPU**.
Total time: about 10 min setup (download + conversion) plus training.

In [ ]:
!nvidia-smi -L
import sys; print(sys.version)

## 1. Get the code and install dependencies

In [ ]:
!test -d Wildcat-Neural-Network || git clone https://github.com/kten314/Wildcat-Neural-Network.git
%cd Wildcat-Neural-Network
!git pull -q
!pip install -q zarr segmentation-models-pytorch

## 2. Download the two data files (about 1.6 GB)
Sizes must come out near 1.6 GB and 6.9 MB. A few hundred bytes means the download failed.

In [ ]:
!mkdir -p data/raw/parihaka
!curl -fL -o data/raw/parihaka/parihaka_labels_train.npz "https://data.mendeley.com/public-files/datasets/gnvyh3msrj/files/0a0ac0af-10ab-47c3-83ac-f38fb0090d4a/file_downloaded"
!curl -fL -o data/raw/parihaka/parihaka_data_train.npz "https://data.mendeley.com/public-files/datasets/gnvyh3msrj/files/2777ae64-68e3-4cb0-aae4-588b30b7009c/file_downloaded"
!ls -lh data/raw/parihaka
!file data/raw/parihaka/*.npz

## 3. Convert to Zarr (about 8 minutes)
This rebuilds the same store you made in Codespaces.

In [ ]:
!python src/convert_parihaka.py

## 4. Smoke test (1-2 minutes)
Checks the whole pipeline runs before the real training. Skip if you are confident.

In [ ]:
!python src/train_unet.py --epochs 1 --steps-per-epoch 10 --tag smoke --figs 1

## 5. Real training run
Defaults: ResNet-34 U-Net, 256x256 patches, 30 epochs x 100 steps. Raise `--epochs` if val mIoU is still climbing at the end.

In [ ]:
!python src/train_unet.py --epochs 30 --steps-per-epoch 100 --batch 16 --tag unet_r34 --seed 0

## 6. Download the results
Then copy the files into your repo's `results/` folder in Codespaces and commit them.

In [ ]:
!zip -qr results_colab.zip results
from google.colab import files
files.download("results_colab.zip")